In [ ]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('data/dataset.csv')

In [8]:
len(df)

114000

In [7]:
len(df.columns)

21

In [10]:
for c in df.columns:
    print(c)

Unnamed: 0
track_id
artists
album_name
track_name
popularity
duration_ms
explicit
danceability
energy
key
loudness
mode
speechiness
acousticness
instrumentalness
liveness
valence
tempo
time_signature
track_genre


In [14]:
df.dtypes

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

In [5]:
df.describe()

,Unnamed: 0,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.000000,114000.000000,1.140000e+05,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,56999.500000,33.238535,2.280292e+05,0.566800,0.641383,5.309140,-8.258960,0.637553,0.084652,0.314910,0.156050,0.213553,0.474068,122.147837,3.904035
std,32909.109681,22.305078,1.072977e+05,0.173542,0.251529,3.559987,5.029337,0.480709,0.105732,0.332523,0.309555,0.190378,0.259261,29.978197,0.432621
min,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,28499.750000,17.000000,1.740660e+05,0.456000,0.472000,2.000000,-10.013000,0.000000,0.035900,0.016900,0.000000,0.098000,0.260000,99.218750,4.000000
50%,56999.500000,35.000000,2.129060e+05,0.580000,0.685000,5.000000,-7.004000,1.000000,0.048900,0.169000,0.000042,0.132000,0.464000,122.017000,4.000000
75%,85499.250000,50.000000,2.615060e+05,0.695000,0.854000,8.000000,-5.003000,1.000000,0.084500,0.598000,0.049000,0.273000,0.683000,140.071000,4.000000
max,113999.000000,100.000000,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5.000000


We are Doing a classification problem here.
We want to predict the genre based on the musical aspects of the song


There are 114000 rows in this dataset
There are 21 Features:
Unnamed: 0 ##This is porbably just an index and can be dropped
track_id ##This is a unique ID
##Song Related##
artists:              object categorical string 
album_name:           object categorical string 
track_name:           object categorical string 
popularity:           int64 0-100
duration_ms:          int64 0-5237295 (87 minutes??)
explicit:             bool
danceability:       float64 0-0.985
energy:             float64 0-1
key                   int64 0-11 (what does this mean?)
loudness            float64 -49.531 - 4.532
mode                  int64 0 or 1
speechiness         float64 0 -.965
acousticness        float64 0 - 0.996
instrumentalness    float64 0 - 1
liveness            float64 0 - 1
valence             float64 0 - .995
tempo               float64 0 - 243.372
time_signature        int64 0 -5
track_genre          object categorical string 

We will not need the artist, album name, or track name. These are not useful categorical variables because how would you encode all those names?
    We can put them away with the unique ID where they will be safe and easy to recall if we need them

Explicit could be a really useful indicator...some genre music tends to be explicit and other genres dont

Popularity is an interesting one. This could be a good indicator if the model is up to date on when it's predicting...maybe a genre of music is really popular at the momement...but if that's the case, is the genre indicating popularity, or is the popularity indicating the genre?

Danceability seems like it would be a great indicator for genre. some genres tend to be danceable, others dont

energy also seems like a great indicator. could really narrow things down at a high level, sort out the death metal from the classical

key, the musical category. Keys seem to be widely used across particular genres, don't they? Seems useful

loudness like energy, could narrow things down a bit

mode, modality, might help. seems better for recommendation than clf.

speechiness seems like a great indicator for things like hip hop where there is rapping, maybe some other niche speaking genres

acousticness, certainly a helpful indicator of genre. some are all accoustic, all electric, or a mix, and those arrangements are key to genre 

instrumentalness, looking for vocals or not. Fully instrumental is a dead giveaway for something like classical, or certain kinds of jazz

liveness, edm is not often performed live

valence positivce or negative vibes. seems real useful for predicting genre, this can ofcourse vary but in general genres keep to a feeling...but then again maybe not?

tempo, yes, seems great for genre indication

time signature, don;t know much about music but seems maybe pretty good? some genres stick to a standard formula, some are known to go off the map.

In [11]:
df

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,...,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,...,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,...,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,...,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music
